# Dependencias e Instalación

## Dependencias

In [ ]:
!uv pip install torch torchaudio torchcodec --torch-backend=auto

## Instalación

In [ ]:
!uv pip install -e /kaggle/input/models/luissantiagobr/coquitts/pytorch/default/1

In [ ]:
!uv pip install -e /kaggle/input/models/luissantiagobr/coquitts/pytorch/default/1/[languages]

## Para evitar incompatibilidades

In [ ]:
!uv pip install transformers==5.0.0

## Para configurar el dataset

In [ ]:
!uv pip install librosa soundfile

# Preparar el dataset

Hay que preparar el dataset con archivos `.wav` y las frases en un mismo `.csv`.

Estructura del dataset:
```
dataset
  ├── metadata.csv
  └── wavs
        ├── example1.wav
        └── example2.wav
```

Información en `metadata.csv`
```
example1|Transcription1
example2|Transcription2
```

> Nota: *Sobre los archivos de audio, de preferencia, para evitar pérdidas de calidad, las grabaciones deberían realizarse en `.wav` con una frecuencia de muestreo de `22050Hz`. Debido a que es lo utilizado para el entrenamiento.*

A continuación se muestra un código en python para crear el archivo `.csv` y remuestrear los `.wav` si es necesario.

In [ ]:
import os
from pathlib import Path
import librosa
import soundfile as sf

def main(input_dir, output_dir, encoding_txt, extension_audio, target_sr=22050):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.is_dir():
        raise ValueError(f"El directorio {input_dir} no existe.")

    # Crear directorio de salida y subcarpeta wavs (si no existen)
    output_dir.mkdir(parents=True, exist_ok=True)
    wavs_dir = output_dir / "wavs"
    wavs_dir.mkdir(exist_ok=True)

    wav_files = list(input_dir.glob(f"*.{extension_audio}"))
    if not wav_files:
        print(f"No se encontraron archivos con extensión .wav en {input_dir}")
        return

    with open(output_dir / "metadata.csv", "w", encoding="utf-8") as out_f:
        for wav_path in wav_files:
            txt_path = wav_path.with_suffix(".txt")
            if not txt_path.exists():
                print(f"No se encuentra transcripción para {wav_path.name}. Se omite.")
                continue

            # Leer transcripción
            text = txt_path.read_text(encoding=encoding_txt).strip()

            # Ruta de salida para el audio (mismo nombre)
            target_wav_path = wavs_dir / wav_path.name

            try:
                # Cargar audio (mono, frecuencia original)
                audio, sr = librosa.load(wav_path, sr=None, mono=True)

                # Resamplear si la frecuencia no es la deseada
                if sr != target_sr:
                    audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)

                # Guardar como WAV (16-bit PCM)
                sf.write(target_wav_path, audio, target_sr, subtype='PCM_16')

                # Escribir metadatos (formato: nombre_sin_ext|texto|texto)
                out_f.write(f"{wav_path.stem}|{text}|{text}\n")

            except Exception as e:
                print(f"Error procesando {wav_path.name}: {e}")
                continue

if __name__ == "__main__":
    input_dir = "/kaggle/input/datasets/luissantiagobr/2024ssv002"
    output_dir = "/kaggle/working/dataset"
    extension_audio = "wav"
    encoding_txt = "windows-1252"

    main(input_dir, output_dir, encoding_txt, extension_audio)
    print("Dataset organizado correctamente")

# Entrenamiento de XTTSv2

## Crear archivo para entrenar

In [ ]:
%%writefile finetunning.py
import os

from trainer import Trainer, TrainerArgs

from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.configs.xtts_config import XttsAudioConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
from TTS.utils.manage import ModelManager

# Logging parameters
RUN_NAME = "GPT_XTTS_v2.0_LJSpeech_FT"
PROJECT_NAME = "XTTS_trainer"
DASHBOARD_LOGGER = "tensorboard"
LOGGER_URI = None

# Set here the path that the checkpoints will be saved. Default: ./run/training/
OUT_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "run", "training")

# Training Parameters
OPTIMIZER_WD_ONLY_ON_WEIGHTS = True  # for multi-gpu training please make it False
START_WITH_EVAL = True  # if True it will star with evaluation
BATCH_SIZE = 3  # set here the batch size
GRAD_ACUMM_STEPS = 84  # set here the grad accumulation steps
# Note: we recommend that BATCH_SIZE * GRAD_ACUMM_STEPS need to be at least 252 for more efficient training. You can increase/decrease BATCH_SIZE but then set GRAD_ACUMM_STEPS accordingly.

# Define here the dataset that you want to use for the fine-tuning on.
config_dataset = BaseDatasetConfig(
    formatter="ljspeech",
    dataset_name="ljspeech",
    path="/kaggle/working/dataset",
    meta_file_train="/kaggle/working/dataset/metadata.csv",
    language="es",
)

# Add here the configs of the datasets
DATASETS_CONFIG_LIST = [config_dataset]

# Define the path where XTTS v2.0.1 files will be downloaded
CHECKPOINTS_OUT_PATH = os.path.join(OUT_PATH, "XTTS_v2.0_original_model_files/")
os.makedirs(CHECKPOINTS_OUT_PATH, exist_ok=True)

# DVAE files
DVAE_CHECKPOINT_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/dvae.pth"
MEL_NORM_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/mel_stats.pth"

# Set the path to the downloaded files
DVAE_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(DVAE_CHECKPOINT_LINK))
MEL_NORM_FILE = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(MEL_NORM_LINK))

# download DVAE files if needed
if not os.path.isfile(DVAE_CHECKPOINT) or not os.path.isfile(MEL_NORM_FILE):
    print(" > Downloading DVAE files!")
    ModelManager._download_model_files([MEL_NORM_LINK, DVAE_CHECKPOINT_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True)


# Download XTTS v2.0 checkpoint if needed
TOKENIZER_FILE_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/vocab.json"
XTTS_CHECKPOINT_LINK = "https://huggingface.co/coqui/XTTS-v2/resolve/main/model.pth"

# XTTS transfer learning parameters: You we need to provide the paths of XTTS model checkpoint that you want to do the fine tuning.
TOKENIZER_FILE = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(TOKENIZER_FILE_LINK))  # vocab.json file
XTTS_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(XTTS_CHECKPOINT_LINK))  # model.pth file

# download XTTS v2.0 files if needed
if not os.path.isfile(TOKENIZER_FILE) or not os.path.isfile(XTTS_CHECKPOINT):
    print(" > Downloading XTTS v2.0 files!")
    ModelManager._download_model_files(
        [TOKENIZER_FILE_LINK, XTTS_CHECKPOINT_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True
    )


# Training sentences generations
SPEAKER_REFERENCE = [
    "/kaggle/working/dataset/wavs/YPMC_300.wav" # speaker reference to be used in training test sentences
]
LANGUAGE = config_dataset.language


def main():
    # init args and config
    model_args = GPTArgs(
        max_conditioning_length=132300,  # 6 secs
        min_conditioning_length=66150,  # 3 secs
        debug_loading_failures=False,
        max_wav_length=441000,  # ~20 seconds
        max_text_length=300,
        mel_norm_file=MEL_NORM_FILE,
        dvae_checkpoint=DVAE_CHECKPOINT,
        xtts_checkpoint=XTTS_CHECKPOINT,  # checkpoint path of the model that you want to fine-tune
        tokenizer_file=TOKENIZER_FILE,
        gpt_num_audio_tokens=1026,
        gpt_start_audio_token=1024,
        gpt_stop_audio_token=1025,
        gpt_use_masking_gt_prompt_approach=True,
        gpt_use_perceiver_resampler=True,
    )
    # define audio config
    audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)
    # training parameters config
    config = GPTTrainerConfig(
        output_path=OUT_PATH,
        model_args=model_args,
        run_name=RUN_NAME,
        project_name=PROJECT_NAME,
        run_description="""
            GPT XTTS training
            """,
        dashboard_logger=DASHBOARD_LOGGER,
        logger_uri=LOGGER_URI,
        audio=audio_config,
        batch_size=BATCH_SIZE,
        batch_group_size=48,
        eval_batch_size=BATCH_SIZE,
        num_loader_workers=4,
        eval_split_max_size=256,
        print_step=50,
        plot_step=500,
        log_model_step=5000,
        save_step=25000,
        save_n_checkpoints=1,
        save_checkpoints=True,
        # target_loss="loss",
        print_eval=False,
        # Optimizer values like tortoise, pytorch implementation with modifications to not apply WD to non-weight parameters.
        optimizer="AdamW",
        optimizer_wd_only_on_weights=OPTIMIZER_WD_ONLY_ON_WEIGHTS,
        optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
        lr=5e-06,  # learning rate
        lr_scheduler="MultiStepLR",
        # it was adjusted accordly for the new step scheme
        lr_scheduler_params={"milestones": [50000 * 18, 150000 * 18, 300000 * 18], "gamma": 0.5, "last_epoch": -1},
        test_sentences=[
            {
                "text": "En contraste con su visión del resto de Centroamérica, presentó un panorama optimista de El Salvador,donde los resultados electorales fueron respetados, lo cual hizo que en su opinión el país sea considerado, en el ámbito internacional, como un lugar de oportunidades.",
                "speaker_wav": SPEAKER_REFERENCE,
                "language": LANGUAGE,
            },
        ],
    )

    # init the model from config
    model = GPTTrainer.init_from_config(config)

    # load training samples
    train_samples, eval_samples = load_tts_samples(
        DATASETS_CONFIG_LIST,
        eval_split=True,
        eval_split_max_size=config.eval_split_max_size,
        eval_split_size=config.eval_split_size,
    )

    # init the trainer and 🚀
    trainer = Trainer(
        TrainerArgs(
            restore_path=None,  # xtts checkpoint is restored via xtts_checkpoint key so no need of restore it using Trainer restore_path parameter
            skip_train_epoch=False,
            start_with_eval=START_WITH_EVAL,
            grad_accum_steps=GRAD_ACUMM_STEPS,
        ),
        config,
        output_path=OUT_PATH,
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
    )
    trainer.fit()


if __name__ == "__main__":
    main()

## Ejecutar entrenamiento

In [ ]:
!CUDA_VISIBLE_DEVICES="0" python finetunning.py

# Inferencia

In [ ]:
import os
import torch
import torchaudio
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# Add here the xtts_config path
CONFIG_PATH = "/kaggle/working/run/training/.../config.json"
# Add here the vocab file that you have used to train the model
TOKENIZER_PATH = "/kaggle/working/run/training/.../vocab.json"
# Add here the checkpoint that you want to do inference with
XTTS_CHECKPOINT = "/kaggle/working/run/training/.../best_model_....pth"
# Add here the speaker reference
SPEAKER_REFERENCE = ["/kaggle/working/dataset/wavs/YPMC_300.wav"]
# output wav path
OUTPUT_WAV_PATH = "/kaggle/working/out.wav"

print("Loading model...")
config = XttsConfig()
config.load_json(CONFIG_PATH)
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_path=XTTS_CHECKPOINT, vocab_path=TOKENIZER_PATH, use_deepspeed=False)
model.cuda() #ACTIVAR PARA GPU

print("Computing speaker latents...")
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(audio_path=SPEAKER_REFERENCE)

In [ ]:
text = """¡Hola!, esta es una voz artificial con el modelo de texto a voz entrenado para el t c u 748."""

print("Inference...")
out = model.inference(
    text,
    "es",
    gpt_cond_latent,
    speaker_embedding,
    temperature=0.95,
    top_k=50,
    top_p=0.8,
    repetition_penalty=2.0,
)
torchaudio.save(OUTPUT_WAV_PATH, torch.tensor(out["wav"]).unsqueeze(0), 24000)

In [ ]:
import IPython

IPython.display.Audio("/kaggle/working/out.wav")